In [14]:
# Bibliothèques tierces
import pandas as pd
import polars as pl

# 0. Flowchart

### 0.1 Sujet Screening , non eligible et eligible a la randomisation 

#### === 0.1.1 Charger les fichiers ===

In [2]:
# import os 
# os.getcwd()

In [ ]:
## DM : Demographie
# Variable demo : 
    # GENDER (form DM )
	# AGE (form DM )
	# ARM (form DM )	
	# MARTIAL ( form SC ) 
	# EDUCYRS ( form SC ) 
	# EMPLOYMENT (form SC ) 
	# BMI (form VS)

In [15]:
DM= pl.concat(
        [
            pl.read_csv("../data/raw/dm1.csv"),
            pl.read_csv("../data/raw/dm2.csv")
        ],
        how="vertical"
    )

In [16]:
dm_counts = (
    DM
    # On garde uniquement ce qui est nécessaire
    .select("STUDYID", "ARMCD")
    # Comptage par bras et par étude
    .group_by(["STUDYID", "ARMCD"])
    .count()
    # Mise en forme IN / OUT
    .pivot(
        index="ARMCD",
        on="STUDYID",
        values="count"
    )
    # Sécurisation si une cellule est absente
    .fill_null(0)
    # Renommage explicite
    .rename({
        "NIDA-CTN-0001": "IN",
        "NIDA-CTN-0002": "OUT"
    })
    # Total par bras
    .with_columns(
        (pl.col("IN") + pl.col("OUT")).alias("Total_bras")
    )
)

dm_counts


C:\Users\Abdo\AppData\Local\Temp\ipykernel_43652\3759412857.py:7: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ARMCD,IN,OUT,Total_bras
str,u32,u32,u32
"""CLON""",36,74,110
"""SCRFAIL""",25,43,68
"""BUPNAL""",77,156,233


In [4]:
totaux = dm_counts.select(
    pl.col("IN").sum().alias("Total_IN"),
    pl.col("OUT").sum().alias("Total_OUT"),
    pl.col("Total_bras").sum().alias("Total_global")
)

print("Total général des patients en screening :", totaux) 

Total général des patients en screening : shape: (1, 3)
┌──────────┬───────────┬──────────────┐
│ Total_IN ┆ Total_OUT ┆ Total_global │
│ ---      ┆ ---       ┆ ---          │
│ u32      ┆ u32       ┆ u32          │
╞══════════╪═══════════╪══════════════╡
│ 138      ┆ 273       ┆ 411          │
└──────────┴───────────┴──────────────┘


#### === 0.1.3 Phase Randomization   ===

In [17]:
# Concaténer IN + OUT et garder seulement les randomisés
arms = ["BUPNAL", "CLON"]

rand_pivot_total = (
    DM
    .filter(pl.col("ARMCD").is_in(arms))
    .group_by(["ARMCD", "STUDYID"])
    .count()
    .pivot(
        values="count",
        index="ARMCD",
        on="STUDYID"
    )
    .rename({
        "NIDA-CTN-0001": "IN",
        "NIDA-CTN-0002": "OUT"
    })
    .with_columns([
        pl.col("IN").fill_null(0),
        pl.col("OUT").fill_null(0),
        (pl.col("IN") + pl.col("OUT")).alias("Total_bras")
    ])
)

print("Total sujets randomisés :", rand_pivot_total["Total_bras"].sum())
rand_pivot_total


Total sujets randomisés : 343


C:\Users\Abdo\AppData\Local\Temp\ipykernel_43652\3379371462.py:8: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ARMCD,OUT,IN,Total_bras
str,u32,u32,u32
"""BUPNAL""",156,77,233
"""CLON""",74,36,110


In [18]:
non_rand_pivot_total = (
    DM
    .filter(~pl.col("ARMCD").is_in(arms))
    .group_by(["ARMCD", "STUDYID"])
    .count()
    .pivot(
        values="count",
        index="ARMCD",
        on="STUDYID"
    )
    .rename({
        "NIDA-CTN-0001": "IN",
        "NIDA-CTN-0002": "OUT"
    })
    .with_columns([
        pl.col("IN").fill_null(0),
        pl.col("OUT").fill_null(0),
        (pl.col("IN") + pl.col("OUT")).alias("Total_bras")
    ])
)

print("Total sujets non randomisés :", non_rand_pivot_total["Total_bras"].sum())
non_rand_pivot_total


Total sujets non randomisés : 68


C:\Users\Abdo\AppData\Local\Temp\ipykernel_43652\1401024169.py:5: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ARMCD,IN,OUT,Total_bras
str,u32,u32,u32
"""SCRFAIL""",25,43,68


## 0.2 Sujet ayant abondonner dans la phase active ( Study day & follow up ) 

#### === 0.2.1 Abondant en phase study day   ===

In [19]:
DS=pl.concat(
        [
            pl.read_csv("../data/raw/ds1.csv"),
            pl.read_csv("../data/raw/ds2.csv")
        ],
        how="vertical"
    )

In [20]:
abandon_sd_total = (
    DS
    # Abandon avant fin de phase active
    .filter(
        (pl.col("EPOCH") != "SCREENING") &
        (pl.col("VISITNUM") < 14) &
        (pl.col("DSDECOD") != "PARTICIPANT COMPLETED ACTIVE PHASE OF STUDY")
    )
    # Une seule ligne par patient
    .unique(subset=["USUBJID"])
    # Garder uniquement les randomisés + récupérer le bras
    .join(
        DM
        .filter(pl.col("ARMCD").is_in(["BUPNAL", "CLON"])),
        on="USUBJID",
        how="inner"
    )
    # Comptage par bras et par étude
    .group_by(["ARMCD", "STUDYID"])
    .count()
    # Pivot IN / OUT
    .pivot(
        values="count",
        index="ARMCD",
        on="STUDYID"
    )
    # Sécurisation + renommage
    .rename({
        "NIDA-CTN-0001": "IN",
        "NIDA-CTN-0002": "OUT"
    })
    .with_columns([
        pl.col("IN").fill_null(0),
        pl.col("OUT").fill_null(0),
        (pl.col("IN") + pl.col("OUT")).alias("Total_bras")
    ])
)

abandon_sd_total


C:\Users\Abdo\AppData\Local\Temp\ipykernel_43652\3102664420.py:20: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ARMCD,OUT,IN,Total_bras
str,u32,u32,u32
"""BUPNAL""",8,6,14
"""CLON""",4,15,19


#### === 0.2.2 Abondant en phase follow up   ===

In [21]:
fu_abandon_total = (
    DS
    # Abandon en follow-up (VISITNUM ≥ 15)
    .filter(
        (pl.col("VISITNUM") > 14) &
        (pl.col("DSDECOD") != "PARTICIPANT COMPLETED ACTIVE PHASE OF STUDY")
    )
    # Une seule ligne par patient
    .unique(subset=["USUBJID"])
    # Joindre pour récupérer le bras randomisé
    .join(
    DM
    .filter(pl.col("ARMCD").is_in(["BUPNAL", "CLON"])),
        on="USUBJID",
        how="inner"
    )
    # Comptage par bras et par étude
    .group_by(["ARMCD", "STUDYID"])
    .count()
    # Pivot IN / OUT
    .pivot(
        values="count",
        index="ARMCD",
        on="STUDYID"
    )
    # Sécurisation + renommage
    .rename({
        "NIDA-CTN-0001": "IN",
        "NIDA-CTN-0002": "OUT"
    })
    .with_columns([
        pl.col("IN").fill_null(0),
        pl.col("OUT").fill_null(0),
        (pl.col("IN") + pl.col("OUT")).alias("Total_bras")
    ])
)

fu_abandon_total


C:\Users\Abdo\AppData\Local\Temp\ipykernel_43652\666173330.py:19: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


ARMCD,IN,OUT,Total_bras
str,u32,u32,u32
"""BUPNAL""",36,59,95
"""CLON""",20,40,60


## 0.3 Analyse en intention de traiter (ITT ) 

In [9]:
# Tous les randomisés sont analysés, même s’ils ont abandonné.
print(rand_pivot_total)

shape: (2, 4)
┌────────┬─────┬─────┬────────────┐
│ ARMCD  ┆ OUT ┆ IN  ┆ Total_bras │
│ ---    ┆ --- ┆ --- ┆ ---        │
│ str    ┆ u32 ┆ u32 ┆ u32        │
╞════════╪═════╪═════╪════════════╡
│ CLON   ┆ 74  ┆ 36  ┆ 110        │
│ BUPNAL ┆ 156 ┆ 77  ┆ 233        │
└────────┴─────┴─────┴────────────┘


## 0.4 Analyse Per Protocol (PP)

##### Le PP inclut uniquement :

* ✔ les patients randomisés
* ✔ qui ont reçu le traitement selon le protocole
* ✔ qui ont completé la période nécessaire pour évaluer le critère principal
* ✔ qui n’ont pas abandonné, ni été perdus, ni eu d’écart majeur au protocole

PPbras​= ITTbras​−Abandons StudyDay<14bras​


Selon le protocole, le critère de jugement principal est : Compléter les 13 jours de détoxification ET avoir un dernier test urinaire négatif aux opiacés le jour 13 ou 14.
Parce que le PP évalue l’efficacité "réelle" du traitement.

In [10]:
#ITT bras , Abandons StudyDay<14bras​
rand_pivot_total, abandon_sd_total

(shape: (2, 4)
 ┌────────┬─────┬─────┬────────────┐
 │ ARMCD  ┆ OUT ┆ IN  ┆ Total_bras │
 │ ---    ┆ --- ┆ --- ┆ ---        │
 │ str    ┆ u32 ┆ u32 ┆ u32        │
 ╞════════╪═════╪═════╪════════════╡
 │ CLON   ┆ 74  ┆ 36  ┆ 110        │
 │ BUPNAL ┆ 156 ┆ 77  ┆ 233        │
 └────────┴─────┴─────┴────────────┘,
 shape: (2, 4)
 ┌────────┬─────┬─────┬────────────┐
 │ ARMCD  ┆ IN  ┆ OUT ┆ Total_bras │
 │ ---    ┆ --- ┆ --- ┆ ---        │
 │ str    ┆ u32 ┆ u32 ┆ u32        │
 ╞════════╪═════╪═════╪════════════╡
 │ CLON   ┆ 15  ┆ 4   ┆ 19         │
 │ BUPNAL ┆ 6   ┆ 8   ┆ 14         │
 └────────┴─────┴─────┴────────────┘)

In [11]:
pp_in_out = (
    rand_pivot_total          # Tableau ITT
    .join(
        abandon_sd_total,     # Tableau abandons SD<14
        on="ARMCD",
        how="left",
        suffix="_abandon"
    )
    .with_columns([
        # Calcul PP_IN = IN_ITT – IN_abandons_SD
        (pl.col("IN") - pl.col("IN_abandon").fill_null(0)).alias("PP_IN"),

        # Calcul PP_OUT = OUT_ITT – OUT_abandons_SD
        (pl.col("OUT") - pl.col("OUT_abandon").fill_null(0)).alias("PP_OUT"),

        # Total PP
        ((pl.col("IN") - pl.col("IN_abandon").fill_null(0)) +
         (pl.col("OUT") - pl.col("OUT_abandon").fill_null(0))).alias("Total_PP")
    ])
    .select(["ARMCD", "PP_IN", "PP_OUT", "Total_PP"])
)

pp_in_out

ARMCD,PP_IN,PP_OUT,Total_PP
str,u32,u32,u32
"""CLON""",21,70,91
"""BUPNAL""",71,148,219
